# Explicabilidad y simulador de riesgo por vano

Hermano de `04_uiti_vano_trayectorias_vano.ipynb`: reutiliza su misma clasificacion
KMeans por vano x ventana (nunca se reajusta aca) y agrega, sobre esa base, un
simulador interactivo de "que pasaria si" a nivel de vano usando el modelo MGCECDL.

**Requiere un kernel Python vivo.** Este cuaderno NO es exportable a HTML estatico ni
publicable como Databricks App, a diferencia de 02/03/04: sus controles
(`ipywidgets`) y el simulador corren en el kernel, no en el navegador.

**Que mide cada mapa (no confundir).** **Criticidad Original** (fila 1) es el grupo
historico: la clase KMeans que 04 ya calculo sobre eventos observados. **Criticidad
Simulada** (fila 2) es la clase que el modelo MGCECDL predice al aplicar las variables
del simulador. Son dos mediciones distintas y nunca comparten leyenda ni titulo.

**Un solo disparador.** El boton "Simular" hace tres cosas de una vez: el mapa
**Criticidad Simulada**, el **Grafo reconstruido** de la seleccion e **Importancia
Variables**. Las tres salen del **mismo** modelo: el MIL entrenado en
`05_mil_vano_ventana`, que puntua BOLSAS -- una bolsa es una celda (vano, ventana) -- y
asigna la clase con `(n_obs observado, u-hat predicho)` sobre la geometria KMeans de 04,
la misma del mapa base. MGCECDL por fila ya no se carga en este cuaderno.

**Requiere dos artefactos de `05_mil_vano_ventana.ipynb`**: `data/models/mil_vano_ventana_v1.pt`
y `data/derived/bolsas_mil_full.joblib`. Los dos estan bajo `data/`, que git ignora: si
faltan, la celda del modelo falla de una con el nombre del cuaderno que los produce.

**Estado actual**: la figura de 4x3 paneles con sus 45 trazas funciona de punta a punta:
los dos mapas a ancho completo, en la fila 3 la importancia de variables, la nube KMeans
con sus fronteras de Voronoi y el grafo reconstruido, y en la fila 4 la evolucion temporal
de los vanos marcados y el reparto de UITI y eventos por grupo. La decision D4 queda
cerrada: la traza que estaba reservada para el grafo es hoy la de sus aristas, con el
indice que siempre tuvo.

In [ ]:
import asyncio
import sys
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError('Este cuaderno requiere ipywidgets para la interfaz interactiva.') from exc
from IPython.display import display

# Sube desde el cwd hasta la raiz del repo (marcada por la carpeta src/), igual que 09.
# Se agregan ROOT y ROOT/src -- no solo src/ -- porque ventanas_015.py importa
# `scripts.extract_geometrias_014` (paquete de nivel de repo, igual que en notebook 10).
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for _path_a_agregar in (ROOT, ROOT / 'src'):
    if str(_path_a_agregar) not in sys.path:
        sys.path.insert(0, str(_path_a_agregar))

# Un kernel que ya importo estos paquetes se queda con la version VIEJA en `sys.modules`:
# "Run All" sin reiniciar NO vuelve a leer el disco. Un rename en src/ -- por ejemplo
# `SelectorVanos._caja` -> `.caja` -- estalla entonces como AttributeError diez celdas mas
# abajo, con el codigo del disco ya correcto. Se purgan ANTES de importarlos, asi el
# cuaderno corre SIEMPRE contra la fuente actual, con o sin reinicio de kernel. Va aca y no
# como `importlib.reload`: reload no rehace los objetos ya construidos con la clase vieja,
# y este cuaderno los reconstruye todos de esta celda para abajo.
for _modulo in [m for m in list(sys.modules)
                if m.split('.')[0] in ('chec_impacto', 'chec_local_interpreter', 'scripts')]:
    del sys.modules[_modulo]

from chec_impacto.data import procesar_dataset_completo
from chec_impacto.models.criticality_assignment import (
    CLAVE_ESPACIO_CANONICO,
    GEOMETRIAS_SHA1_ESPERADO,
    cargar_geometria_014,
    verificar_sha1_geometrias,
)
from chec_impacto.data.bags import cargar_bolsas
from chec_impacto.models.mil_persistencia import cargar_modelo_mil
from chec_impacto.training import resolve_training_device
from chec_local_interpreter.mil_simulador_015 import (
    gates_de_bolsas,
    grafo_de_gates,
    seleccionar_bolsas,
    simular_bolsas,
    trazas_grafo,
)
from chec_local_interpreter.vano_controls import build_knobs, expand_knob_overrides
from chec_local_interpreter.vano_widgets import (
    construir_selector_casillas,
    construir_selector_vanos,
)
from chec_local_interpreter.ventanas_015 import (
    capas_mapa_historico,
    cargar_clases_desde_014,
    centro_y_zoom,
    construir_hist_class_cache,
    construir_mask_cache,
    construir_tabla_vano_ventana,
    construir_ventanas,
    fid_de_punto,
    frontera_kmeans,
    nube_fondo,
    nube_seleccion,
    reparto_por_clase,
    series_temporal_vanos,
)
from scripts.extract_geometrias_014 import (
    DEFAULT_NOTEBOOK_PATH,
    DEFAULT_OUTPUT_PATH,
    extraer_geometrias_014,
)

# Sonda del contrato que rompio el cuaderno dos veces. Si el kernel siguiera sirviendo una
# version vieja de vano_widgets, falla ACA -- primera celda, mensaje que dice que hacer --
# en vez de a los 10 minutos de procesamiento, en la celda del panel.
_sonda = construir_selector_vanos(['0'])
assert hasattr(_sonda, 'caja'), (
    'vano_widgets viejo en memoria: el selector de casillas sin `.caja`. '
    'Reinicia el kernel. '
    f'(modulo cargado desde {sys.modules["chec_local_interpreter.vano_widgets"].__file__})'
)
del _sonda

In [ ]:
# Ventana climatica igual que 03_mgcecdl_training / 09_simulador: cambiarla generaria un
# set de features distinto al que el modelo cargado en la celda SEAM espera.
VENTANA_CLIMATICA_HORAS = 12
CLAVE_ESPACIO = CLAVE_ESPACIO_CANONICO  # '2' -- espacio canonico fijado en criticality_assignment.py
DEVICE = resolve_training_device('auto')

# Misma paleta que 01.4: los grupos historicos de este cuaderno SON los de 01.4, nunca se
# reajustan, asi que el color tiene que significar lo mismo en los dos cuadernos.
NOMBRES_GRUPOS = ['Bajo', 'Medio', 'Medio-Alto', 'Alto']
COLORES_GRUPOS = ['rgb(252,187,161)', 'rgb(251,106,74)', 'rgb(203,24,29)', 'rgb(103,0,13)']
# UN solo codigo de ausencia en los DOS mapas: negro, el `COLOR_SIN_EVENTO` de 01.4.
# El vano sin eventos en la ventana no tiene clase, y la ausencia no es la clase mas baja;
# que el base lo pintara gris y el simulado negro obligaba a recordar dos codigos para la
# misma cosa. Los cuatro colores KMeans significan lo mismo en los dos mapas.
COLOR_SIN_EVENTO = 'rgb(0,0,0)'
COLOR_MARCADO = '#0072b2'
# Equipos: mismos colores que 01.4, por el mismo motivo que la paleta de grupos --
# un naranja tiene que seguir siendo un transformador al pasar de un cuaderno a otro.
COLOR_TRAFO = '#f59e0b'
COLOR_SWITCH = '#7c3aed'
ANCHO_MAPA = 3.0
ANCHO_MAPA_MARCADO = round(ANCHO_MAPA * 1.4, 2)

# Fila 1, paridad 01.4: un vano MARCADO se dibuja con el color de SU clase, sobre un halo
# blanco que lo despega del fondo (01.4: `width=ANCHO_MAPA_RESALTE * 2.6, color='white'`).
# Un color plano de "seleccionado" encima de la clase congela lo que se ve: la ventana
# cambia la clase por debajo y el vano marcado sigue igual en pantalla. COLOR_MARCADO
# queda solo para la fila 2, donde la clase la pone el modelo y no el KMeans.
COLOR_HALO = 'white'
ANCHO_HALO = round(ANCHO_MAPA_MARCADO * 2.6, 2)
OPACIDAD_NUBE = 0.45               # 01.4, para que la nube de fondo no tape el resaltado
OPACIDAD_FRONTERA = 0.28           # 01.4: el contorno es fondo, no dato
# Paleta de 01.4 para las series por vano: apta para daltonismo y distinta de la escala de
# grupos, porque aca el color identifica AL VANO, no a su clase.
COLORES_VANOS = ['#0072b2', '#009e73', '#cc79a7', '#56b4e9', '#e69f00', '#8c564b']
N_CUPOS_EVOLUCION = len(COLORES_VANOS)
# Paleta de los MODOS de variable en el grafo. Deliberadamente fuera de la familia de los
# grupos KMeans (rojos/naranjas) y de los equipos: un rojo en el mapa y un rojo en el grafo
# significarian cosas sin ninguna relacion. Se recorre en el orden de las modalidades del
# artefacto.
PALETA_MODALIDADES = ['#0d9488', '#be185d']   # verde azulado y rosa oscuro


In [ ]:
# --- Reutilizacion de la geometria KMeans de 01.4 (design section F) -------
# Falla RAPIDO aca, antes de procesar el dataset completo (celda siguiente): si 01.4 fue
# editado y sus centroides se movieron, no tiene sentido esperar el procesamiento pesado
# para enterarse. `cargar_clases_desde_014` (celda 7, via hist_class_cache) repite esta
# misma verificacion por cada ventana consultada -- barata, y evita que una geometria
# cacheada quede sin recomprobar dentro de la misma sesion.
GEOMETRIAS_PATH = DEFAULT_OUTPUT_PATH
if not GEOMETRIAS_PATH.exists():
    extraer_geometrias_014(DEFAULT_NOTEBOOK_PATH, GEOMETRIAS_PATH)
_sha1_real, _coincide = verificar_sha1_geometrias(GEOMETRIAS_PATH, esperado=GEOMETRIAS_SHA1_ESPERADO)
assert _coincide, (
    f'La geometria KMeans extraida de 01.4 no coincide con la esperada '
    f'(esperado={GEOMETRIAS_SHA1_ESPERADO}, real={_sha1_real}). 01.4 fue modificado; '
    f'01.5 depende de esa geometria.'
)
# La geometria en si (no solo su sha1): la nube KMeans de la fila 3 tiene que dibujarse en
# el MISMO espacio en que se asignan las clases -- el canonico '2' es (log_x=False,
# log_y=True). Leerlo de la geometria y no fijarlo a mano evita que un cambio de espacio
# deje la nube en ejes que ya no corresponden a las fronteras.
GEOMETRIA_014 = cargar_geometria_014(GEOMETRIAS_PATH, CLAVE_ESPACIO)
print(f'Geometria 01.4 verificada -- sha1 coincide ({_sha1_real[:12]}...) | '
      f'espacio {CLAVE_ESPACIO}: log_x={GEOMETRIA_014.logs[0]}, log_y={GEOMETRIA_014.logs[1]}')

In [ ]:
DATA_PATH = ROOT / 'data' / 'Indicadores_vano_v3.csv'
VARIABLES_SELECCION_PATH = ROOT / 'data' / 'Variables_seleccion.xlsx'
MODEL_DIR = ROOT / 'data' / 'models'

# Mismo preprocesamiento real usado en entrenamiento (03_mgcecdl_training / 09_simulador):
# sin muestreo ni filtro de UITI, para que context_df quede alineado FILA A FILA con X --
# la clave que permite reusar la MISMA mascara (circuito, ventana) para el mapa historico
# (sin modelo) y, en un PR futuro, para las predicciones del modelo sobre esas mismas filas.
datos = procesar_dataset_completo(
    path_clima=DATA_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target='UITI_VANO',
    filtro_uiti_max=None,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

feature_names = list(datos['features'])
X_raw_model = np.asarray(datos['X'], dtype=np.float32)
Xdf = datos['Xdata'].copy().reset_index(drop=True)
context_df = datos['df_original_copy'].copy().reset_index(drop=True)
label_encoders = datos.get('label_encoders', {})
max_values_imputed = datos.get('max_values_imputed', {})

# Ya no se construye el escalador min-max de MGCECDL. El simulador y la importancia de
# variables corren sobre el modelo MIL del cuaderno 05, cuya matriz de instancias es RAW,
# asi que `preparar_splits_estratificados` + `escalar_features_minmax_mgcecdl` -- de lo
# mas caro de esta celda -- no alimentaban ya a nadie.
assert len(context_df) == len(X_raw_model), (
    'context_df y X_raw_model deben quedar alineados fila a fila'
)
print(f'{len(context_df):,} filas | {len(feature_names)} features')

In [ ]:
# --- UN solo modelo: el MIL por bolsas del cuaderno 05 (cierra el SEAM D1) ------------
# El tablero entero -- mapa "Criticidad Simulada", grafo reconstruido e "Importancia
# Variables" -- responde a este modelo y a esta unidad: la BOLSA (vano x ventana), que es
# la unidad en la que 04 define la criticidad. MGCECDL por fila salio del cuaderno: tener
# dos modelos contestando paneles vecinos del mismo tablero significaba que el panel y el
# mapa hablaban de cosas distintas sin que nada en pantalla lo dijera.
# Requiere DOS artefactos que produce el cuaderno 05 y que viven bajo `data/` (ignorado
# por git): el modelo y el cache de bolsas. Falla ACA, con el nombre del cuaderno que los
# genera, en vez de a los diez minutos en la celda del boton.
RUTA_MODELO_MIL = MODEL_DIR / 'mil_vano_ventana_v1.pt'
RUTA_BOLSAS_MIL = ROOT / 'data' / 'derived' / 'bolsas_mil_full.joblib'
for _ruta in (RUTA_MODELO_MIL, RUTA_BOLSAS_MIL):
    assert _ruta.exists(), (
        f'Falta {_ruta.name}: lo produce 05_mil_vano_ventana.ipynb. Corre ese cuaderno '
        'antes que este.'
    )

BOLSAS = cargar_bolsas(RUTA_BOLSAS_MIL)
X_INST, FEATURES_MIL, BAG_INDEX = BOLSAS['X'], BOLSAS['features'], BOLSAS['bag_index']
# `device='cpu'` a proposito y no DEVICE: una seleccion son decenas de instancias, asi que
# el traslado a GPU/MPS cuesta mas de lo que ahorra, y saca una variable de dtype de en
# medio de un camino interactivo.
MIL = cargar_modelo_mil(RUTA_MODELO_MIL, device='cpu', features_esperadas=FEATURES_MIL)

# La guarda que hace comparables los dos mapas: si el MIL se hubiera entrenado con OTRA
# geometria KMeans, sus clases usarian los mismos 4 colores para significar otra cosa.
for _campo in ('offset', 'scale', 'centroides'):
    assert np.allclose(getattr(MIL.geometria, _campo), getattr(GEOMETRIA_014, _campo)), (
        f'La geometria del modelo MIL difiere de la de 01.4 en {_campo}: sus clases NO son '
        'las del mapa base y no pueden compartir la paleta.'
    )
assert tuple(MIL.geometria.logs) == tuple(GEOMETRIA_014.logs)

# Las 70 primeras features del MIL son exactamente las de MGCECDL (22 estaticas + 48 de
# clima); las 10 restantes son COD_CAUSA y sus indicadores, que no son controles del
# simulador. Por eso el catalogo de knobs de la celda siguiente sirve para los dos.
assert list(FEATURES_MIL[:len(feature_names)]) == list(feature_names), (
    'Las features del MIL ya no empiezan por las de MGCECDL: el catalogo de knobs '
    'apuntaria a columnas equivocadas.'
)
# Los MODOS de variable con los que el modelo agrupa las columnas -- son los mismos que
# usa la fusion FiLM (el clima reescala lo estructural), asi que colorear los nodos del
# grafo por modalidad muestra exactamente la particion que el modelo usa por dentro.
COLUMNAS_MODALIDAD = {m: set(int(i) for i in idx)
                      for m, idx in MIL.model.base.modality_feature_indices.items()}
MODALIDADES_MIL = list(COLUMNAS_MODALIDAD)
assert len(MODALIDADES_MIL) == len(PALETA_MODALIDADES), (
    f'El artefacto trae {len(MODALIDADES_MIL)} modalidades y la figura tiene trazas de '
    f'nodo para {len(PALETA_MODALIDADES)}: agrega la traza que falta antes de seguir.'
)
COLORES_MODALIDAD = dict(zip(MODALIDADES_MIL, PALETA_MODALIDADES))

print(f'MIL cargado -- {len(BAG_INDEX.keys):,} bolsas | {X_INST.shape[0]:,} instancias x '
      f'{len(FEATURES_MIL)} features | geometria identica a 01.4')
print('modos de variable: ' + ' | '.join(
    f'{m} ({len(COLUMNAS_MODALIDAD[m])})' for m in MODALIDADES_MIL))

In [ ]:
# --- construir_ventanas + per-(vano, ventana) events + caches (design section A) -------
VENTANAS = construir_ventanas(context_df['FECHA'])
TABLA = construir_tabla_vano_ventana(context_df, VENTANAS)
mask_para = construir_mask_cache(TABLA)
clases_para = construir_hist_class_cache(TABLA, mask_para)

# La clase de CADA celda (vano x ventana) de una sola pasada: es la misma asignacion por
# centroide mas cercano que hace `clases_para` ventana por ventana (es puntual, fila a
# fila), pero calculada una vez para poder dibujar la nube KMeans completa de la fila 3.
CLASE_TABLA, _n_clamped = cargar_clases_desde_014(
    TABLA['num_eventos'].to_numpy(dtype=float),
    TABLA['uiti_acumulado'].to_numpy(dtype=float),
)
# La nube de fondo va SUBMUESTREADA (ver `nube_fondo`): las 111 mil celdas completas son
# 1,2 MB de coordenadas en una sola rafaga por el comm del widget, por encima del
# `iopub_data_rate_limit` de 1 MB/s que ipykernel trae por defecto -- y un mensaje que se
# pasa de ese limite se descarta, con lo que la figura no llega a dibujarse.
NUBE_FONDO = nube_fondo(TABLA, CLASE_TABLA)
# Extension FIJA del plano (eventos, UITI), como en 01.4: los ejes y la frontera no
# dependen de la seleccion, solo del dataset.
EXTENSION = [float(TABLA['num_eventos'].min()), float(TABLA['num_eventos'].max()),
             float(TABLA['uiti_acumulado'].min()), float(TABLA['uiti_acumulado'].max())]
_n_nube = sum(len(c['x']) for c in NUBE_FONDO)
print(f'nube KMeans: {len(TABLA):,} celdas | {_n_nube:,} dibujadas (muestra fija) | '
      f'por clase {[len(c["x"]) for c in NUBE_FONDO]} | '
      f'{_n_clamped} valores recortados por eps')

CIRCUITOS = sorted(TABLA['CIRCUITO'].astype(str).unique())
VANOS_POR_CIRCUITO = {
    c: sorted(g['FID_VANO'].unique().tolist())
    for c, g in TABLA.groupby(TABLA['CIRCUITO'].astype(str))
}

print(f'{len(TABLA):,} celdas vano x ventana con eventos | {len(VENTANAS)} ventanas | '
      f'{TABLA["FID_VANO"].nunique():,} vanos distintos | {len(CIRCUITOS)} circuitos')


# Geometria FISICA de cada vano (no confundir con la geometria KMeans de la celda 4): mismo
# shapefile y mismo join que el mapa de 01.3/01.4. No se extrae a src/ porque es solo
# lectura + reindexado geoespacial, sin logica propia que valga la pena testear por fuera
# de lo que TABLA/capas_mapa_historico ya cubren.
def _norm_id(serie):
    return (serie.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
            .replace({'': pd.NA, '<NA>': pd.NA, 'nan': pd.NA, 'None': pd.NA}))


_lineas = gpd.read_file(ROOT / 'data' / 'GEO' / 'MVLINSEC.shp')
if str(_lineas.crs) != 'EPSG:4326':
    _lineas = _lineas.to_crs('EPSG:4326')
_lineas['FID_VANO_GEO'] = _norm_id(_lineas['G3E_FID'])
_utiles = _lineas[_lineas['CIRCUITO'].astype(str).isin(set(CIRCUITOS))]

GEO_POR_CIRCUITO = {}
for _c, _g in _utiles.groupby(_utiles['CIRCUITO'].astype(str)):
    fids, lats, lons = [], [], []
    for _fid, _geom in zip(_g['FID_VANO_GEO'], _g.geometry):
        if _geom is None or _geom.is_empty:
            continue
        for _p in ([_geom] if _geom.geom_type == 'LineString' else list(getattr(_geom, 'geoms', []))):
            xs, ys = _p.xy
            fids.append(str(_fid))
            lats.append([round(v, 5) for v in ys])
            lons.append([round(v, 5) for v in xs])
    if fids:
        # `bounds` es lo que permite encuadrar el mapa sobre el circuito elegido, igual
        # que 01.4: [lat_min, lat_max, lon_min, lon_max].
        _la = [v for l in lats for v in l]
        _lo = [v for l in lons for v in l]
        GEO_POR_CIRCUITO[_c] = {
            'fids': fids, 'lat': lats, 'lon': lons,
            'bounds': [round(min(_la), 5), round(max(_la), 5),
                       round(min(_lo), 5), round(max(_lo), 5)],
        }


def _equipo(nombre):
    """Transformadores e interruptores del circuito, igual que 01.4 celda 5. Si el
    shapefile no esta, el mapa se dibuja sin equipos en vez de fallar: son contexto
    de lectura, no el dato del tablero."""
    ruta = ROOT / 'data' / 'GEO' / nombre
    if not ruta.exists():
        return {}
    g = gpd.read_file(ruta)
    if str(g.crs) != 'EPSG:4326':
        g = g.to_crs('EPSG:4326')
    g = g[g['CIRCUITO'].astype(str).isin(set(CIRCUITOS))]
    g = g[g.geometry.notna() & ~g.geometry.is_empty]
    return {c: {'lat': [round(float(p.y), 5) for p in gg.geometry],
                'lon': [round(float(p.x), 5) for p in gg.geometry]}
            for c, gg in g.groupby(g['CIRCUITO'].astype(str))}


TRAFOS = _equipo('GDBCHEC_TRANSFOR.shp')
SWITCHES = _equipo('SWITCHES.shp')

# UITI y eventos por vano y ventana: solo alimentan el hover, igual que 01.4. El grupo
# NO se guarda aca -- sale de `clases_para`, que es la unica fuente de clases.
DATOS_VENTANA = [{} for _ in VENTANAS]
for _fid, _vi, _u, _n in zip(TABLA['FID_VANO'], TABLA['ventana_i'],
                             TABLA['uiti_acumulado'], TABLA['num_eventos']):
    DATOS_VENTANA[int(_vi)][str(_fid)] = (float(_u), int(_n))

print(f'{len(GEO_POR_CIRCUITO)} circuitos con geometria fisica | '
      f'{sum(len(v["lat"]) for v in TRAFOS.values()):,} transformadores | '
      f'{sum(len(v["lat"]) for v in SWITCHES.values()):,} switches')

In [ ]:
KNOBS = build_knobs(
    feature_names=feature_names,
    original_feature_df=Xdf,
    label_encoders=label_encoders,
    max_values_imputed=max_values_imputed,
)
print(f'{len(KNOBS)} controles (Knob catalog, PR2a) -- '
      f'{sum(1 for k in KNOBS if k.kind == "categorical")} categoricos, '
      f'{sum(1 for k in KNOBS if k.kind == "numeric")} numericos, '
      f'{sum(1 for k in KNOBS if k.kind == "constant")} constantes')

In [ ]:
# --- Inventario de trazas CONGELADO (design section G). Los indices 0-17 no se mueven:
# la grilla cambio de 2x3 a 3x2 y ninguna traza cambio de posicion en la LISTA, solo de
# subplot. El grafo reconstruido (decision D4) sigue mutando el indice 7 en un PR futuro,
# con sus trazas nuevas a partir del 18.
# Los mapas ocupan las DOS columnas de su fila (colspan=2): un mapa geografico compartido
# entre 4 filas de vanos y una barra de importancia no se lee, y estirarlo al ancho
# completo es lo unico que permite distinguir tramos vecinos. Las otras dos figuras --
# importancia de variables y el panel reservado -- bajan juntas a la fila 3.
# Los equipos (14-17) van ultimos por orden de dibujo: los marcadores tienen que quedar
# por encima de las lineas, como en 01.4.
IDX = {
    'clases': [0, 1, 2, 3],          # fila 1 (2 columnas) -- mapa historico (01.4), PR3
    'sin_dato': 4,                    # fila 1 -- sin eventos en la ventana
    'marcados': 5,                    # fila 1 -- halo de vanos marcados
    'ranking': 6,                     # fila 3 col 1 -- importancia de variables, PR4
    'grafo_aristas': 7,               # fila 3 col 3 -- grafo reconstruido (decision D4)
    'pred_clases': [8, 9, 10, 11],    # fila 2 (2 columnas) -- mapa predicho MGCECDL, PR5
    'pred_sin_dato': 12,               # fila 2, PR5
    'pred_marcados': 13,               # fila 2, PR5
    'trafos': 14,                      # fila 1 -- equipos, PR6
    'switches': 15,                    # fila 1 -- equipos, PR6
    'pred_trafos': 16,                 # fila 2 -- equipos, PR6
    'pred_switches': 17,               # fila 2 -- equipos, PR6
    # Nuevas, a partir del 18 y sin mover ninguna anterior (design section G).
    'marcados_clases': [18, 19, 20, 21],  # fila 1 -- marcado, con el color de SU clase
    'marcados_sin_dato': 22,              # fila 1 -- marcado sin celda en la ventana: negro
    'nube_clases': [23, 24, 25, 26],      # fila 3 col 2 -- nube KMeans (fondo fijo)
    'nube_seleccion': 27,                 # fila 3 col 2 -- celdas de lo marcado
    'grafo_pesos': 28,                    # fila 3 col 3 -- peso de cada arista
    'grafo_nodos': [29, 45],              # fila 3 col 3 -- variables, una traza por modo
    'frontera': 30,                       # fila 3 col 2 -- Voronoi de las fronteras KMeans
    'evolucion': [31, 32, 33, 34, 35, 36],  # fila 4 col 1 -- una serie por vano marcado
    'violin_uiti': [37, 38, 39, 40],      # fila 4 col 2 -- reparto de UITI por grupo
    'violin_eventos': [41, 42, 43, 44],   # fila 4 col 3 -- reparto de eventos por grupo
}

_fig = make_subplots(
    rows=4, cols=3,
    specs=[[{'type': 'map', 'colspan': 3}, None, None],
           [{'type': 'map', 'colspan': 3}, None, None],
           [{'type': 'xy'}, {'type': 'xy'}, {'type': 'xy'}],
           [{'type': 'xy'}, {'type': 'xy'}, {'type': 'xy'}]],
    # Un titulo por subplot REAL: las celdas `None` del colspan no consumen ninguno.
    subplot_titles=(
        'Criticidad Original',
        'Criticidad Simulada',
        'Importancia Variables',
        'Grupos KMeans de vanos',
        'Grafo reconstruido',
        'Evolucion temporal de los vanos marcados',
        'UITI por grupo',
        'Eventos por grupo',
    ),
    row_heights=[0.29, 0.29, 0.21, 0.21],
    horizontal_spacing=0.07, vertical_spacing=0.06,
)

for _clase in range(4):                                          # 0-3
    _fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase],
        legendgroup='hist', legendgrouptitle_text='Criticidad original',
        line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), row=1, col=1)
_fig.add_trace(go.Scattermap(                                    # 4
    lat=[], lon=[], mode='lines', name='Sin evento en la ventana', legendgroup='hist',
    line=dict(width=ANCHO_MAPA, color=COLOR_SIN_EVENTO),
    hovertext=[], hoverinfo='text',
), row=1, col=1)
# El marcado de la fila 1 va en DOS capas, como en 01.4: primero el halo blanco ancho
# (esta traza, la 5), y despues -- indices 18-22, al final por orden de dibujo -- la linea
# con el color de la clase del vano. El halo no va a la leyenda: una linea blanca sobre
# fondo blanco no dice nada ahi.
_fig.add_trace(go.Scattermap(                                    # 5
    lat=[], lon=[], mode='lines', name='Vano marcado', legendgroup='hist',
    showlegend=False,
    line=dict(width=ANCHO_HALO, color=COLOR_HALO),
    hovertext=[], hoverinfo='text',
), row=1, col=1)

_fig.add_trace(go.Bar(x=[], y=[], orientation='h', showlegend=False,
                      hovertext=[], hoverinfo='text'), row=3, col=1)  # 6
# La barra ya no muestra la magnitud cruda sino su participacion softmax: un porcentaje
# se lee sin conocer las unidades del modelo, una "sensibilidad min-max" de 0.0143 no.
_fig.update_xaxes(title_text='Relevancia (softmax)', tickformat='.0%', row=3, col=1)
# `type='category'`: con la traza vacia un eje numerico inventa marcas 0..4 que no
# significan nada. Un eje de categorias vacio no dibuja ninguna.
_fig.update_yaxes(type='category', tickfont=dict(size=10), row=3, col=1)

# La 7 era la traza RESERVADA para el grafo reconstruido (decision D4). Este PR la puebla:
# pasa a ser el trazo de las aristas. Su indice nunca se movio, que era el punto de
# haberla dejado ahi desde el principio.
_fig.add_trace(go.Scatter(
    x=[], y=[], mode='lines', showlegend=False,
    line=dict(width=1.0, color='rgba(120,110,110,0.45)'), hoverinfo='skip',
), row=3, col=3)  # 7

for _clase in range(4):                                          # 8-11
    _fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase],
        legendgroup='pred', legendgrouptitle_text='Criticidad simulada', showlegend=False,
        line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), row=2, col=1)
_fig.add_trace(go.Scattermap(                                    # 12
    lat=[], lon=[], mode='lines', name='Sin evento / no simulado', legendgroup='pred',
    showlegend=False, line=dict(width=ANCHO_MAPA, color=COLOR_SIN_EVENTO),
    hovertext=[], hoverinfo='text',
), row=2, col=1)
_fig.add_trace(go.Scattermap(                                    # 13
    lat=[], lon=[], mode='lines', name='Vano marcado', legendgroup='pred', showlegend=False,
    line=dict(width=ANCHO_MAPA_MARCADO, color=COLOR_MARCADO),
    hovertext=[], hoverinfo='text',
), row=2, col=1)

# Los equipos van al final para dibujarse ENCIMA de los tramos. Se repiten por fila
# porque una traza pertenece a un solo subplot: no hay forma de compartirla entre los
# dos mapas, y sin ellos la fila 2 se leeria como otra geografia.
for _fila, _leyenda in ((1, True), (2, False)):
    for _nombre, _color, _tam in [('Transformadores', COLOR_TRAFO, 6),
                                  ('Switches', COLOR_SWITCH, 5)]:
        _fig.add_trace(go.Scattermap(                             # 14-17
            lat=[], lon=[], mode='markers', name=_nombre,
            legendgroup='equipos', legendgrouptitle_text='Equipos',
            showlegend=_leyenda,
            marker=dict(size=_tam, color=_color), hovertext=[], hoverinfo='text',
        ), row=_fila, col=1)

# --- 18-22: el marcado de la fila 1, con el color de su clase (paridad 01.4) ---------
for _clase in range(4):                                          # 18-21
    _fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase],
        legendgroup='hist', showlegend=False,
        line=dict(width=ANCHO_MAPA_MARCADO, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), row=1, col=1)
_fig.add_trace(go.Scattermap(                                    # 22
    lat=[], lon=[], mode='lines', name='Marcado sin eventos', legendgroup='hist',
    line=dict(width=ANCHO_MAPA_MARCADO, color=COLOR_SIN_EVENTO),
    hovertext=[], hoverinfo='text',
), row=1, col=1)

# --- 23-27: la nube KMeans de 01.4 en el panel que estaba reservado ------------------
# El fondo son TODAS las celdas (vano x ventana) del dataset, agrupadas por clase: es
# donde estan las fronteras, y no se mueve nunca. Encima, las celdas de lo marcado en la
# ventana activa. Ahi se ve lo que el mapa solo insinua: mover la ventana mueve el punto
# del vano por el plano (eventos, UITI) y por eso cambia su clase.
for _clase in range(4):                                          # 23-26
    _fig.add_trace(go.Scattergl(
        x=[], y=[], mode='markers', name=NOMBRES_GRUPOS[_clase],
        legendgroup='nube', legendgrouptitle_text='Nube KMeans', showlegend=False,
        marker=dict(size=3.5, color=COLORES_GRUPOS[_clase], opacity=OPACIDAD_NUBE),
        hoverinfo='skip',
    ), row=3, col=2)
_fig.add_trace(go.Scattergl(                                     # 27
    x=[], y=[], mode='markers', name='Seleccion', showlegend=False,
    marker=dict(size=9, color=[], line=dict(width=1.4, color='#111111')),
    hovertext=[], hoverinfo='text',
), row=3, col=2)
_fig.update_xaxes(title_text='Eventos en la ventana',
                  type='log' if GEOMETRIA_014.logs[0] else 'linear', row=3, col=2)
_fig.update_yaxes(title_text='UITI acumulado',
                  type='log' if GEOMETRIA_014.logs[1] else 'linear', row=3, col=2)

# --- 28-29: el grafo reconstruido de la seleccion (decision D4) ----------------------
# El peso viaja en un marcador en el PUNTO MEDIO de cada arista y no en el ancho de la
# linea: una sola traza de lineas no puede variar su ancho por segmento, y partirla en
# una traza por arista serian 64 trazas que hay que restilar una por una.
_fig.add_trace(go.Scattergl(                                     # 28
    x=[], y=[], mode='markers', showlegend=False,
    marker=dict(size=[], color=[], colorscale='Reds', cmin=0.0, showscale=False,
                line=dict(width=0.4, color='#5b4a48')),
    hovertext=[], hoverinfo='text',
), row=3, col=3)
# Los nodos van en UNA traza por modo de variable, no en una sola con vector de colores:
# asi cada modo entra por si mismo a la leyenda, que es donde se lee que significa el
# color. La segunda traza (45) se crea al final del inventario, para no correr ningun
# indice existente -- por eso hay un constructor compartido y dos llamadas separadas.
def _traza_nodos_grafo(modalidad):
    return go.Scatter(
        x=[], y=[], mode='markers+text', name=modalidad,
        legendgroup='grafo', legendgrouptitle_text='Modo de variable',
        marker=dict(size=7, color=COLORES_MODALIDAD[modalidad],
                    line=dict(width=0.5, color='#1f2937')),
        text=[], textposition='middle right', textfont=dict(size=7, color='#334155'),
        hovertext=[], hoverinfo='text',
    )


_fig.add_trace(_traza_nodos_grafo(MODALIDADES_MIL[0]), row=3, col=3)   # 29
# Sin ejes: un grafo en disposicion circular no mide nada en x ni en y. El rango se fija a
# mano y con holgura -- sin ella los rotulos de los nodos del borde salen cortados.
_fig.update_xaxes(visible=False, showticklabels=False, range=[-1.75, 1.75], row=3, col=3)
_fig.update_yaxes(visible=False, showticklabels=False, range=[-1.3, 1.3], row=3, col=3)
_fig.add_annotation(text='', xref='x3 domain', yref='y3 domain', x=0.5, y=0.5,
                    showarrow=False, align='center',
                    font=dict(size=11, color='#7a5c58'))
IDX_ANOTACION_GRAFO = len(_fig.layout.annotations) - 1

# --- 30: frontera KMeans (Voronoi) DEBAJO de la nube --------------------------------
# Misma escala escalonada y misma opacidad de 01.4. Va como `Contour` y no como trazas
# de linea porque Plotly dibuja contornos en una capa POR DEBAJO de los scatter del mismo
# subplot, sin importar el orden de las trazas: la nube queda encima aunque esta traza sea
# la 30.
ESCALA_CONTORNO = []
for _g, _color in enumerate(COLORES_GRUPOS):
    ESCALA_CONTORNO.append([_g / 4.0, _color])
    ESCALA_CONTORNO.append([(_g + 1) / 4.0, _color])
_fig.add_trace(go.Contour(                                       # 30
    z=[[0, 0], [0, 0]], x=[0, 1], y=[0, 1], colorscale=ESCALA_CONTORNO,
    zmin=-0.5, zmax=3.5, showscale=False, opacity=OPACIDAD_FRONTERA, hoverinfo='skip',
    line=dict(width=1.2, color='rgba(120,20,20,0.6)'),
    contours=dict(start=-0.5, end=3.5, size=1, coloring='fill'), showlegend=False,
), row=3, col=2)

# --- 31-36: evolucion temporal, un CUPO por vano marcado ----------------------------
# Cupos fijos y no una traza por vano: el inventario de trazas esta congelado, y un
# circuito puede tener cientos de vanos marcados. Se dibujan los primeros
# N_CUPOS_EVOLUCION y el panel dice cuantos quedaron afuera, igual que 01.4.
for _cupo in range(N_CUPOS_EVOLUCION):                           # 31-36
    _fig.add_trace(go.Scatter(
        x=[], y=[], mode='lines+markers', name='', showlegend=False,
        line=dict(color=COLORES_VANOS[_cupo], width=2), marker=dict(size=5),
        connectgaps=False,  # el hueco de una ventana sin celda NO se cose
        hovertext=[], hoverinfo='text',
    ), row=4, col=1)
_fig.update_xaxes(title_text='Ventana', tickmode='array',
                  tickvals=[v['i'] for v in VENTANAS],
                  ticktext=[v['etiqueta'] for v in VENTANAS],
                  tickfont=dict(size=9), row=4, col=1)
_fig.update_yaxes(title_text='UITI acumulado',
                  type='log' if GEOMETRIA_014.logs[1] else 'linear', row=4, col=1)

# --- 37-44: violines por grupo (UITI y eventos) -------------------------------------
for _fila_violin, _columna, _campo in ((37, 2, 'UITI'), (41, 3, 'Eventos')):
    for _clase in range(4):                                      # 37-40 y 41-44
        _fig.add_trace(go.Violin(
            y=[], name=NOMBRES_GRUPOS[_clase], showlegend=False,
            fillcolor=COLORES_GRUPOS[_clase], opacity=0.85,
            line=dict(color='#5b4a48', width=1),
            box_visible=True, meanline_visible=False, points=False, spanmode='hard',
            hovertemplate=f'{_campo} -- %{{x}}: %{{y:,.2f}}<extra></extra>',
        ), row=4, col=_columna)
_fig.update_yaxes(title_text='UITI acumulado',
                  type='log' if GEOMETRIA_014.logs[1] else 'linear', row=4, col=2)
_fig.update_yaxes(title_text='Eventos', rangemode='tozero', row=4, col=3)

_fig.add_trace(_traza_nodos_grafo(MODALIDADES_MIL[1]), row=3, col=3)   # 45
_fig.update_xaxes(tickfont=dict(size=9), row=4, col=2)
_fig.update_xaxes(tickfont=dict(size=9), row=4, col=3)

# El aviso del mapa simulado va en coordenadas de PAPEL y no de eje: un subplot de tipo
# `map` no tiene ejes cartesianos a los que anclar una anotacion. El centro sale del
# dominio que `make_subplots` ya calculo, asi cambiar `row_heights` no lo desalinea.
_dominio_simulado = _fig.layout.map2.domain
_fig.add_annotation(
    text='', xref='paper', yref='paper',
    x=(_dominio_simulado.x[0] + _dominio_simulado.x[1]) / 2.0,
    y=(_dominio_simulado.y[0] + _dominio_simulado.y[1]) / 2.0,
    showarrow=False, align='center', font=dict(size=13, color='#5b4a48'),
    bgcolor='rgba(255,255,255,0.88)', bordercolor='#e4c4c0', borderwidth=1, borderpad=8,
)
IDX_ANOTACION_SIMULADO = len(_fig.layout.annotations) - 1

_fig.update_layout(
    map=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10),
    map2=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10),
    title=dict(text='Simulador Criticidad'),
    # Dos mapas apilados a ancho completo piden alto: con los 760 de la grilla 2x3 cada
    # mapa quedaba en una franja de ~300 px y los tramos se pisaban entre si.
    height=1560, width=1280, template='plotly_white',
    legend=dict(y=1.0, yanchor='top'),
)

# Los indices se verifican al generar, igual que 01.3/01.4: si alguien reordena las
# trazas esto falla ACA, no se descubre silenciosamente en la celda de dibujo.
assert len(_fig.data) == 46, len(_fig.data)
assert all(_fig.data[i].type == 'scattermap' for i in IDX['clases'] + [IDX['sin_dato'], IDX['marcados']])
assert [_fig.data[i].line.color for i in IDX['clases']] == COLORES_GRUPOS
assert _fig.data[IDX['ranking']].type == 'bar'
assert _fig.data[IDX['grafo_aristas']].type == 'scatter'
assert _fig.data[IDX['grafo_aristas']].mode == 'lines'
assert all(_fig.data[i].type == 'scattermap'
           for i in IDX['pred_clases'] + [IDX['pred_sin_dato'], IDX['pred_marcados']])
# Los equipos son PUNTOS y van despues de todas las lineas: si alguien los adelanta,
# quedan tapados por los tramos y esto falla al generar, no en el navegador.
assert all(_fig.data[i].mode == 'markers'
           for i in (IDX['trafos'], IDX['switches'], IDX['pred_trafos'], IDX['pred_switches']))
assert min(IDX['trafos'], IDX['switches'], IDX['pred_trafos'], IDX['pred_switches']) > IDX['pred_marcados']
# El marcado con color de clase va DESPUES del halo blanco, o el halo lo taparia.
assert min(IDX['marcados_clases']) > IDX['marcados']
assert [_fig.data[i].line.color for i in IDX['marcados_clases']] == COLORES_GRUPOS
assert _fig.data[IDX['marcados_sin_dato']].line.color == COLOR_SIN_EVENTO
# El MISMO negro para la ausencia en los dos mapas: si alguien los separa, esto falla al
# generar la figura y no en la lectura de un tablero ya publicado.
assert (_fig.data[IDX['sin_dato']].line.color
        == _fig.data[IDX['pred_sin_dato']].line.color == COLOR_SIN_EVENTO)
assert all(_fig.data[i].type == 'scattergl' for i in IDX['nube_clases'] + [IDX['nube_seleccion']])
assert _fig.data[IDX['frontera']].type == 'contour'
assert all(_fig.data[i].type == 'violin' for i in IDX['violin_uiti'] + IDX['violin_eventos'])
assert [_fig.data[i].fillcolor for i in IDX['violin_uiti']] == COLORES_GRUPOS
assert len(IDX['evolucion']) == N_CUPOS_EVOLUCION
assert [_fig.data[i].name for i in IDX['grafo_nodos']] == MODALIDADES_MIL
assert ([_fig.data[i].marker.color for i in IDX['grafo_nodos']]
        == [COLORES_MODALIDAD[m] for m in MODALIDADES_MIL])

fig = go.FigureWidget(_fig)
print(f'FigureWidget con {len(fig.data)} trazas (indices 0-13 congelados, design section G)')

In [ ]:
# --- Fila 1: mapa historico con paridad 01.4 + seleccion por casilla o por clic ------
# Tres cosas que el mapa de 01.4 hace y este no hacia: se ENCUADRA sobre el circuito
# elegido (sin eso el circuito queda como un garabato diminuto en un mapa centrado en
# Manizales), dibuja transformadores e interruptores, y da hover por tramo. La cuarta es
# la seleccion: en 01.4 un vano se marca con su casilla O tocandolo en el mapa, y las dos
# vias son EL MISMO estado -- el clic alterna la casilla y deja que todo se rehaga desde
# ahi. Un registro paralelo es como la lista, el mapa y el ranking empiezan a contar
# cosas distintas.


def _seleccion_actual():
    return circuito_widget.value, ventana_widget.value, set(vano_widget.value)


def _capas_de_la_seleccion(clases_por_fid, *, campo, nombres_clase):
    """Las capas de UN mapa, con las etiquetas y el customdata que necesita el clic.

    `campo` nombra en el tooltip a que pertenece la clase -- "Criticidad original" en
    la fila 1, "Criticidad simulada" en la fila 2. Esa distincion vive ahora en el tooltip de
    cada tramo y en la leyenda, que es donde se lee mientras se mira el mapa, en vez de
    en un parrafo fijo al costado del panel.
    """
    circuito, ventana_i, marcados = _seleccion_actual()
    geo = GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []})
    ventana = VENTANAS[ventana_i]
    datos = DATOS_VENTANA[ventana_i]

    etiquetas = {}
    for fid in geo['fids']:
        uiti, eventos = datos.get(fid, (0.0, 0))
        clase = clases_por_fid.get(fid)
        # Sin celda en la ventana no hay clase, y eso NO es el grupo mas bajo: es la
        # ausencia del dato. Mismo criterio que el tooltip de 01.4.
        etiquetas[fid] = (
            f'<b>Vano {fid}</b><br>{ventana["etiqueta"]}: {ventana["periodo"]}'
            f'<br>{campo}: {nombres_clase[clase] if clase is not None else "sin dato"}'
            f'<br>UITI acumulado: {uiti}<br>Eventos: {eventos}'
            + ('<br>(marcado)' if fid in marcados else '')
        )
    return capas_mapa_historico(geo, clases_por_fid, marcados=marcados,
                                etiquetas_por_fid=etiquetas)


def _volcar_capa(traza, capa):
    """Las cuatro columnas van juntas SIEMPRE: si `customdata` se desfasa de lat/lon,
    Plotly desalinea el resto de la traza y el clic devuelve el vano equivocado."""
    traza.lat = capa['lat']
    traza.lon = capa['lon']
    traza.hovertext = capa['hovertext']
    traza.customdata = capa['customdata']


def _redibujar_mapa_historico(*_ignorado):
    circuito, ventana_i, _marcados = _seleccion_actual()
    capas = _capas_de_la_seleccion(clases_para(circuito, ventana_i),
                                   campo='Criticidad original', nombres_clase=NOMBRES_GRUPOS)
    mask_ventana = mask_para(circuito, ventana_i)
    seleccion = nube_seleccion(TABLA, CLASE_TABLA, mask_ventana=mask_ventana,
                               marcados=_marcados)
    # Los violines describen SOLO los vanos marcados (regla de 01.4, ver
    # `reparto_por_clase`); la evolucion, los primeros N_CUPOS_EVOLUCION de ellos.
    reparto = reparto_por_clase(TABLA, CLASE_TABLA, mask_ventana=mask_ventana,
                                marcados=_marcados)
    marcados_ordenados = [f for f in GEO_POR_CIRCUITO.get(circuito, {}).get('fids', [])
                          if f in _marcados]
    marcados_ordenados = list(dict.fromkeys(marcados_ordenados))[:N_CUPOS_EVOLUCION]
    series = series_temporal_vanos(TABLA, circuito=circuito, fids=marcados_ordenados,
                                   n_ventanas=len(VENTANAS))
    with fig.batch_update():
        for _clase in range(4):
            _volcar_capa(fig.data[IDX['clases'][_clase]], capas['clases'][_clase])
            _volcar_capa(fig.data[IDX['marcados_clases'][_clase]],
                         capas['marcados_por_clase'][_clase])
        _volcar_capa(fig.data[IDX['sin_dato']], capas['sin_dato'])
        _volcar_capa(fig.data[IDX['marcados']], capas['marcados'])
        _volcar_capa(fig.data[IDX['marcados_sin_dato']], capas['marcados_sin_dato'])
        # La nube: solo el resaltado se repinta, el fondo se dibuja una vez al arrancar.
        _traza_nube = fig.data[IDX['nube_seleccion']]
        _traza_nube.x = seleccion['x']
        _traza_nube.y = seleccion['y']
        _traza_nube.marker.color = [COLORES_GRUPOS[c] for c in seleccion['clase']]
        _traza_nube.hovertext = [
            f'<b>Vano {f}</b><br>Eventos: {x:,}<br>UITI acumulado: {y}'
            f'<br>Grupo: {NOMBRES_GRUPOS[c]}'
            for f, x, y, c in zip(seleccion['fid'], seleccion['x'],
                                  seleccion['y'], seleccion['clase'])
        ]
        # Fila 4: evolucion por cupo y violines por grupo.
        for _cupo, _traza in enumerate(fig.data[i] for i in IDX['evolucion']):
            _serie = series[_cupo] if _cupo < len(series) else None
            _traza.x = _serie['x'] if _serie else []
            _traza.y = _serie['uiti'] if _serie else []
            _traza.hovertext = ([
                f'<b>Vano {_serie["fid"]}</b><br>{VENTANAS[i]["etiqueta"]}: '
                f'{VENTANAS[i]["periodo"]}<br>UITI: {u}<br>Eventos: {e}'
                for i, u, e in zip(_serie['x'], _serie['uiti'], _serie['eventos'])
            ] if _serie else [])
        for _clase in range(4):
            fig.data[IDX['violin_uiti'][_clase]].y = reparto[_clase]['uiti']
            fig.data[IDX['violin_eventos'][_clase]].y = reparto[_clase]['eventos']
        # Mismo motivo que el eje de relevancia: sin vanos marcados el eje lineal de
        # eventos autoescala a [-1, 4] y muestra un "-1 eventos" que no existe.
        _max_eventos = max((e for g in reparto for e in g['eventos']), default=0)
        fig.update_yaxes(range=[0, _max_eventos * 1.15 if _max_eventos else 5],
                         row=4, col=3)


def _pintar_circuito(*_ignorado):
    """Lo que depende del CIRCUITO y no de la ventana: equipos y encuadre. Se separa del
    repintado por ventana porque mover la ventana no tiene por que recentrar el mapa --
    en 01.4 el encuadre tambien se hace una sola vez por circuito (`ULTIMO_CENTRADO`)."""
    circuito = circuito_widget.value
    tr = TRAFOS.get(circuito, {'lat': [], 'lon': []})
    sw = SWITCHES.get(circuito, {'lat': [], 'lon': []})
    vista = centro_y_zoom(GEO_POR_CIRCUITO.get(circuito, {}).get('bounds'))
    with fig.batch_update():
        # Solo la fila 1: los equipos de la fila 2 los pinta el mapa simulado, que antes
        # de la primera simulacion no muestra NADA.
        for _i_tr, _i_sw in ((IDX['trafos'], IDX['switches']),):
            fig.data[_i_tr].lat, fig.data[_i_tr].lon = tr['lat'], tr['lon']
            fig.data[_i_tr].hovertext = ['<b>Transformador</b>'] * len(tr['lat'])
            fig.data[_i_sw].lat, fig.data[_i_sw].lon = sw['lat'], sw['lon']
            fig.data[_i_sw].hovertext = ['<b>Interruptor / switch</b>'] * len(sw['lat'])
        if vista is not None:
            # Los dos mapas comparten encuadre a proposito: la comparacion fila 1 contra
            # fila 2 solo se sostiene si las dos miran exactamente la misma geografia.
            for _mapa in ('map', 'map2'):
                getattr(fig.layout, _mapa).center = vista['center']
                getattr(fig.layout, _mapa).zoom = vista['zoom']


_DESC = {'description_width': 'initial'}  # sin esto ipywidgets trunca los rotulos
circuito_widget = widgets.Dropdown(options=CIRCUITOS, description='Circuito',
                                   style=_DESC)
# El rotulo lleva las fechas del intervalo y no solo "V1": una ventana sin sus fechas
# obliga a ir a buscar a que periodo corresponde cada vez que se mueve el deslizador.
ventana_widget = widgets.SelectionSlider(
    options=[(f'{v["etiqueta"]}: {v["periodo"]}', v['i']) for v in VENTANAS],
    description='Ventana', continuous_update=False, style=_DESC,
    layout=widgets.Layout(width='560px'),
)
# Casillas, no SelectMultiple: es la unica forma de que un clic en el mapa alterne el
# MISMO control que el usuario ve, y de que marcar un vano no borre los ya marcados.
vano_widget = construir_selector_vanos(VANOS_POR_CIRCUITO.get(circuito_widget.value, []))


# "Marcar todos" / "Desmarcar", igual que el par de botones de 04: con circuitos de
# cientos de vanos, marcarlos de a uno no es una opcion. Los dos van por el selector y no
# por un registro propio, asi que emiten UN solo cambio de `value` y disparan un solo
# repintado -- no uno por casilla.
boton_marcar_todos = widgets.Button(description='Marcar todos', button_style='')
boton_desmarcar = widgets.Button(description='Desmarcar', button_style='')
boton_marcar_todos.on_click(lambda _b: vano_widget.marcar_todos())
boton_desmarcar.on_click(lambda _b: vano_widget.desmarcar_todos())


def _on_circuito_change(_change):
    vano_widget.poblar(VANOS_POR_CIRCUITO.get(circuito_widget.value, []))
    _pintar_circuito()
    _redibujar_mapa_historico()


def _al_hacer_clic(traza, puntos, _estado):
    """Un clic sobre un tramo alterna su vano. El fid sale de `customdata` y no del
    indice del punto: los tramos viajan concatenados con un `None` de separador, asi que
    ese indice cambia con la ventana."""
    fid = fid_de_punto(traza.customdata, getattr(puntos, 'point_inds', ()) or ())
    if fid is not None:
        vano_widget.alternar(fid)


# SOLO el mapa base. La fila 2 es la SALIDA del modelo, no un control: marcar un vano
# desde ahi mezcla "lo que yo elegi" con "lo que el modelo predijo" sobre la misma
# superficie, que es justo la confusion que separa a las dos filas (D2).
# Nota sobre el alcance del clic: plotly solo convierte un clic en evento si en ese punto
# hay hover, y en un `scattermap` de lineas el hover se calcula contra los VERTICES del
# tramo (`scattermap/hover.js`: distancia por punto, radio minimo 3 px, tope
# `layout.hoverdistance`). Hay que tocar el tramo cerca de uno de sus quiebres, no en
# cualquier parte del segmento. `hoverdistance` sube de los 20 px por defecto a 30 para
# que el blanco sea mas generoso sin llegar a marcar un vano lejano.
for _i_traza in IDX['clases'] + [IDX['sin_dato'], IDX['marcados']]:
    fig.data[_i_traza].on_click(_al_hacer_clic)
fig.layout.hoverdistance = 30

# Tier 0 del presupuesto de interactividad (design section A): elegir circuito, mover la
# ventana o marcar un vano no llama al modelo -- sin debounce ni epoch guard, que
# pertenecen al tier 1/2 (fila 2, ranking, boton "Simular"), fuera del alcance de este PR.
circuito_widget.observe(_on_circuito_change, names='value')
ventana_widget.observe(_redibujar_mapa_historico, names='value')
vano_widget.observe(_redibujar_mapa_historico, names='value')

# El fondo de la nube y la frontera van una sola vez: no dependen de la seleccion (01.4
# ajusta el KMeans una vez y elegir circuito o vanos solo cambia que se resalta).
FRONTERA = frontera_kmeans(GEOMETRIA_014, x_min=EXTENSION[0], x_max=EXTENSION[1],
                           y_min=EXTENSION[2], y_max=EXTENSION[3])
with fig.batch_update():
    for _clase in range(4):
        fig.data[IDX['nube_clases'][_clase]].x = NUBE_FONDO[_clase]['x']
        fig.data[IDX['nube_clases'][_clase]].y = NUBE_FONDO[_clase]['y']
    fig.data[IDX['frontera']].x = FRONTERA['x']
    fig.data[IDX['frontera']].y = FRONTERA['y']
    fig.data[IDX['frontera']].z = FRONTERA['z']

_pintar_circuito()               # equipos y encuadre del circuito inicial
_redibujar_mapa_historico()      # primer dibujo, con la seleccion inicial

In [ ]:
# --- Importancia de variables, fila 3 col 1 (design section A, decision D7) ------------
# Barrido de sensibilidad min-max sobre el MISMO modelo y la MISMA unidad que el mapa
# simulado: la bolsa (vano, ventana) del cuaderno 05. Antes corria sobre MGCECDL por
# fila, y entonces el panel y el mapa hablaban de cosas distintas -- el panel media el
# efecto de una variable sobre filas de evento sueltas, el mapa mostraba clases de bolsa
# -- sin que nada en pantalla lo dijera.
# Sigue sin ser SHAP (decision D5): es un barrido min-max, y asi se llama en todo el
# cuaderno. Corre UNA vez, dentro del mismo job del boton "Simular" (celda siguiente) y
# bajo la misma epoca: un solo disparador significa que mapa, grafo e importancia siempre
# describen la MISMA seleccion. Sin vanos marcados el grano es el circuito completo en esa
# ventana, las mismas bolsas que pinta el mapa.
from chec_local_interpreter.mil_simulador_015 import construir_relevance_cache_mil

rankear_relevancia = construir_relevance_cache_mil(
    predictor=MIL,
    X_inst=X_INST,
    bag_index=BAG_INDEX,
    feature_names=FEATURES_MIL,
    knobs=KNOBS,
    label_encoders=label_encoders,
    max_values_imputed=max_values_imputed,
)

RANKING_VACIO = {'vacio': True, 'filas': [], 'n_vanos': 0, 'n_filas': 0, 'mensaje': None}


def _calcular_ranking(circuito, ventana_i, marcados):
    """La parte pesada: `1 + 2 x knobs_numericos` pasadas de bolsas (una base compartida
    y el min/max de cada control). Se llama DENTRO del job de "Simular" para que una sola
    epoca cubra mapa, grafo e importancia. Medido sobre el modelo real: 0,09 s para 24
    bolsas y 0,34 s para 537, asi que alcanza con el LRU de sesion y no hace falta el
    cache en disco que necesitaba la version por filas."""
    return rankear_relevancia(circuito, VENTANAS[ventana_i]['etiqueta'], marcados)


def _pintar_ranking(resultado):
    """Repaint puro, cero pasadas del modelo."""
    # Orden ascendente: en un bar horizontal Plotly dibuja la primera categoria abajo, asi
    # que la variable mas relevante (primera en `filas`, ya ordenada descendente por
    # magnitud cruda) queda arriba.
    filas = list(reversed(resultado['filas']))
    # El rango se fija a mano: con la traza vacia Plotly autoescala a [-1, 4] y el
    # `tickformat` de porcentaje lo muestra como "0% ... 600%", que se lee como si algo
    # tuviera 600% de relevancia. Con datos se ajusta al maximo real, que ademas aprovecha
    # el ancho del panel sin tocar los valores.
    _tope = max((fila['relevancia'] for fila in filas), default=0.0)
    with fig.batch_update():
        fig.update_xaxes(range=[0, _tope * 1.15 if _tope > 0 else 0.25], row=3, col=1)
        fig.data[IDX['ranking']].y = [fila['label'] for fila in filas]
        fig.data[IDX['ranking']].x = [fila['relevancia'] for fila in filas]
        # La magnitud cruda no desaparece: baja al hover, que es donde se la consulta
        # cuando hace falta comparar contra otra corrida.
        fig.data[IDX['ranking']].hovertext = [
            f'<b>{fila["label"]}</b><br>Relevancia: {fila["relevancia"]:.1%}'
            f'<br>Sensibilidad min-max: {fila["magnitud_max_cambio_abs"]:.4g}'
            for fila in filas
        ]


_pintar_ranking(RANKING_VACIO)  # vacio hasta el primer "Simular"


In [ ]:
# --- Fila 2: mapa "Criticidad Simulada" + boton "Simular" (design section A, decision D2)
# El boton es el UNICO disparador y hace TRES cosas de una sola vez, bajo la misma epoca:
# el mapa simulado, el grafo reconstruido de la seleccion y el barrido de importancia de
# la celda anterior. Ya no hay alternador base/simulado/delta: el mapa de la fila 2
# muestra SIEMPRE la clase simulada, que es lo que el boton promete.
#
# El mapa y el grafo salen del modelo MIL del cuaderno 05, que puntua BOLSAS: una bolsa es
# una celda (vano, ventana) y su clase sale de `asignar_clase(n_obs OBSERVADO, u-hat
# predicho)` sobre la geometria KMeans de 01.4 -- la MISMA con la que se pinta el mapa
# base, que es por lo que los dos mapas comparten paleta por construccion y no por
# convencion. `n_obs` nunca se simula: es un eje del espacio que define la clase.
# Debounce asincronico (design section A): `asyncio.ensure_future` + cancelacion en el
# propio event loop del kernel, NUNCA `threading.Timer` -- ipykernel enruta la salida de
# los widgets con el parent header thread-local, asi que una escritura desde un hilo en
# segundo plano cae en la celda equivocada. `_EPOCA` es el guard de epoca: cualquier evento
# que invalide un job en vuelo la avanza y la escritura tardia se descarta.
from chec_local_interpreter.vano_app_015 import (
    DEBOUNCE_SEGUNDOS,
    ESTADO_SIMULADO,
    aplicar_si_vigente,
    clases_por_fid_para_estado,
    siguiente_epoca,
)
from chec_local_interpreter.vano_widgets import widget_for_knob

# El estado vacio se PIDE a la funcion en vez de escribirlo a mano: escrito a mano se
# desincroniza en cuanto `trazas_grafo` agrega una columna, que es exactamente lo que
# paso al sumarle el indice de modalidad a cada nodo.
GRAFO_VACIO = trazas_grafo(np.zeros((1, 1)), [''])

_EPOCA = 0
_tarea_pendiente_simular = None
_ultimo_resultado_simulacion = None   # DataFrame de simulate_explicit_overrides, o None
_ultima_seleccion_simulada = None     # (circuito, ventana_i) al que corresponde ese resultado

STATUS = widgets.HTML(
    'Sin simular todavia -- elige variables (opcional) y presiona "Simular".'
)

_knobs_por_id = {k.id: k for k in KNOBS}
# Casillas y no `SelectMultiple`, por el mismo motivo que la lista de vanos: en un
# `SelectMultiple` un clic sin ctrl borra todo lo ya elegido, y aca justamente se quiere
# simular VARIAS variables a la vez. Cada casilla es independiente y `value` sigue siendo
# la tupla de knob ids, asi que `_reconstruir_controles_knob` no se entera del cambio.
knob_selector_widget = construir_selector_casillas(
    [(k.label, k.id) for k in KNOBS], titulo='', alto='150px', ancho_casilla='230px',
    layout=widgets.Layout(width='100%'),
)
controles_knob_box = widgets.VBox([])
_controles_knob_actuales = {}


def _reconstruir_controles_knob(_change=None):
    global _controles_knob_actuales
    _controles_knob_actuales = {
        knob_id: widget_for_knob(_knobs_por_id[knob_id]) for knob_id in knob_selector_widget.value
    }
    controles_knob_box.children = list(_controles_knob_actuales.values())


knob_selector_widget.observe(_reconstruir_controles_knob, names='value')

boton_simular = widgets.Button(description='Simular', button_style='primary')


_CAPA_VACIA = {'lat': [], 'lon': [], 'hovertext': [], 'customdata': []}


def _redibujar_mapa_predicho(*_ignorado):
    """Repaint puro, CERO llamadas al modelo.

    Antes de la primera simulacion de la seleccion activa el mapa no se dibuja: ni
    tramos, ni equipos, ni leyenda -- solo el aviso de que hay que presionar "Simular".
    Un mapa completo pintado de "aun no simulado" ocupa el mismo lugar y tiene la misma
    forma que un resultado, y esa es justamente la confusion que la fila 2 no puede
    permitirse (D2).

    Con resultado: color de grupo KMeans para lo que el simulador predijo -- la MISMA
    paleta del mapa base, porque es la misma geometria -- y NEGRO para todo lo demas
    (vano sin evento en la ventana, o no seleccionado), igual que la estructura del
    circuito en 01.4.
    """
    circuito, ventana_i, _marcados = _seleccion_actual()
    hay_resultado = (
        _ultimo_resultado_simulacion is not None
        and _ultima_seleccion_simulada == (circuito, ventana_i)
    )
    if not hay_resultado:
        with fig.batch_update():
            for _i in (IDX['pred_clases'] + [IDX['pred_sin_dato'], IDX['pred_marcados'],
                                             IDX['pred_trafos'], IDX['pred_switches']]):
                fig.data[_i].lat, fig.data[_i].lon = [], []
                fig.data[_i].hovertext = []
                fig.data[_i].showlegend = False
            fig.layout.annotations[IDX_ANOTACION_SIMULADO].text = (
                'El mapa simulado aparece al presionar <b>Simular</b>.'
            )
        return

    clases_por_fid = clases_por_fid_para_estado(_ultimo_resultado_simulacion, ESTADO_SIMULADO)
    capas = _capas_de_la_seleccion(clases_por_fid, campo='Criticidad simulada',
                                   nombres_clase=NOMBRES_GRUPOS)
    tr = TRAFOS.get(circuito, {'lat': [], 'lon': []})
    sw = SWITCHES.get(circuito, {'lat': [], 'lon': []})
    with fig.batch_update():
        for _clase in range(4):
            _volcar_capa(fig.data[IDX['pred_clases'][_clase]], capas['clases'][_clase])
            fig.data[IDX['pred_clases'][_clase]].showlegend = True
        # Negro: sin evento en la ventana, o fuera de la seleccion simulada. Un vano que
        # el simulador no puntuo no tiene clase, y la ausencia no es la clase mas baja.
        _volcar_capa(fig.data[IDX['pred_sin_dato']], capas['sin_dato'])
        fig.data[IDX['pred_sin_dato']].name = 'Sin evento / no simulado'
        fig.data[IDX['pred_sin_dato']].line.color = COLOR_SIN_EVENTO
        fig.data[IDX['pred_sin_dato']].showlegend = True
        # 13 queda vacia a proposito: en este mapa lo coloreado ES la seleccion, asi que
        # un halo de "marcado" encima no distingue nada que el color no diga ya.
        _volcar_capa(fig.data[IDX['pred_marcados']], _CAPA_VACIA)
        fig.data[IDX['pred_marcados']].showlegend = False
        fig.data[IDX['pred_trafos']].lat, fig.data[IDX['pred_trafos']].lon = tr['lat'], tr['lon']
        fig.data[IDX['pred_trafos']].hovertext = ['<b>Transformador</b>'] * len(tr['lat'])
        fig.data[IDX['pred_switches']].lat, fig.data[IDX['pred_switches']].lon = sw['lat'], sw['lon']
        fig.data[IDX['pred_switches']].hovertext = ['<b>Interruptor / switch</b>'] * len(sw['lat'])
        fig.layout.annotations[IDX_ANOTACION_SIMULADO].text = ''


def _limpiar_resultado_simulacion(_change=None):
    """Circuito o ventana cambiaron: el ultimo resultado ya NO corresponde a la
    seleccion activa -- se descarta (fila 2 vuelve a "Aun no simulado" y el panel de
    importancia se vacia) en vez de mostrar la corrida de OTRA seleccion, que violaria
    la regla anti-confusion (D2)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada, _EPOCA
    _ultimo_resultado_simulacion = None
    _ultima_seleccion_simulada = None
    _EPOCA = siguiente_epoca(_EPOCA)  # invalida cualquier job en vuelo
    _redibujar_mapa_predicho()
    _pintar_grafo(None)
    _pintar_ranking(RANKING_VACIO)


def _pintar_grafo(grafo):
    """Repaint puro del panel del grafo. Un grafo ANULADO no se dibuja a medias: se
    vacian las trazas y se dice por que. `estadistico_colapso` anula cuando las
    compuertas no varian entre vanos -- y su veredicto incluye `effective_rank <= 1`,
    que con menos de 3 vanos se cumple por construccion (la matriz centrada de 1 o 2
    filas tiene rango 1). Dibujar igual seria presentar el grafo experto FIJO como si
    lo hubiera estimado esta seleccion."""
    if grafo is None:
        trazas, mensaje = GRAFO_VACIO, 'Presiona "Simular" para estimar el grafo.'
    elif grafo['voided']:
        trazas = GRAFO_VACIO
        mensaje = (f'Grafo no estimable: las compuertas no varian entre los '
                   f'{grafo["n_vanos"]} vanos de la seleccion.<br>'
                   '<sup>Hacen falta al menos 3 vanos con comportamiento distinto.</sup>')
    else:
        trazas, mensaje = trazas_grafo(grafo['matriz'], FEATURES_MIL), ''

    with fig.batch_update():
        fig.data[IDX['grafo_aristas']].x = trazas['aristas']['x']
        fig.data[IDX['grafo_aristas']].y = trazas['aristas']['y']
        _pesos = trazas['pesos']
        fig.data[IDX['grafo_pesos']].x = _pesos['x']
        fig.data[IDX['grafo_pesos']].y = _pesos['y']
        fig.data[IDX['grafo_pesos']].hovertext = _pesos['hovertext']
        # El tamano codifica el peso relativo DE ESTA seleccion: los pesos absolutos
        # cambian dos ordenes de magnitud entre ventanas y un tamano fijo por valor
        # dejaria el panel vacio o saturado segun cual se mire.
        _maximo = max(_pesos['peso'], default=0.0) or 1.0
        fig.data[IDX['grafo_pesos']].marker.size = [4 + 10 * (p / _maximo) for p in _pesos['peso']]
        fig.data[IDX['grafo_pesos']].marker.color = list(_pesos['peso'])
        # Un nodo por variable, con su NOMBRE al lado y el color de su modo. El rotulo
        # se manda hacia afuera del circulo (a la derecha en la mitad derecha, a la
        # izquierda en la izquierda) para que no se monte sobre las aristas.
        _nodos = trazas['nodos']
        for _i_traza, _modalidad in zip(IDX['grafo_nodos'], MODALIDADES_MIL):
            _cuales = [k for k, col in enumerate(_nodos['indice'])
                       if col in COLUMNAS_MODALIDAD[_modalidad]]
            _traza_nodo = fig.data[_i_traza]
            _traza_nodo.x = [_nodos['x'][k] for k in _cuales]
            _traza_nodo.y = [_nodos['y'][k] for k in _cuales]
            _traza_nodo.text = [_nodos['texto'][k] for k in _cuales]
            _traza_nodo.textposition = ['middle right' if _nodos['x'][k] >= 0
                                        else 'middle left' for k in _cuales]
            _traza_nodo.hovertext = [f'<b>{_nodos["texto"][k]}</b><br>Modo: {_modalidad}'
                                     for k in _cuales]
        fig.layout.annotations[IDX_ANOTACION_GRAFO].text = mensaje


def _simular(epoca_job):
    """Computo pesado -- bloqueante dentro de la corutina (design section A: un job ya
    iniciado no se puede interrumpir). Mapa simulado, grafo e importancia, en ese orden y
    en el mismo job. Guarda y repinta SOLO si `epoca_job` sigue vigente al terminar
    (epoch guard)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
    circuito, ventana_i, marcados = _seleccion_actual()
    seleccion = seleccionar_bolsas(BAG_INDEX, circuito=circuito,
                                   ventana=VENTANAS[ventana_i]['etiqueta'],
                                   marcados=marcados)
    if seleccion['n_bolsas'] == 0:
        aplicar_si_vigente(
            lambda: setattr(STATUS, 'value',
                            'Sin bolsas (vano x ventana) para esta seleccion.'),
            epoca_job=epoca_job, epoca_actual=lambda: _EPOCA,
        )
        return

    overrides = expand_knob_overrides(
        {knob_id: widget.value for knob_id, widget in _controles_knob_actuales.items()}, KNOBS,
    )

    t0 = time.perf_counter()
    resultado, metadata = simular_bolsas(
        MIL, X_INST, seleccion=seleccion, feature_names=FEATURES_MIL, overrides=overrides,
        label_encoders=label_encoders, max_values_imputed=max_values_imputed,
    )
    # El grafo se estima sobre las features OBSERVADAS de la seleccion, no sobre las
    # simuladas: describe a estos vanos, no al escenario hipotetico.
    gates = gates_de_bolsas(MIL, X_INST[seleccion['filas']], seleccion['instance_bag'],
                            seleccion['n_bolsas'])
    grafo = grafo_de_gates(gates, MIL.model.edge_index, n_features=len(FEATURES_MIL))
    ranking = _calcular_ranking(circuito, ventana_i, marcados)
    duracion = time.perf_counter() - t0

    def _escribir():
        global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
        _ultimo_resultado_simulacion = resultado
        _ultima_seleccion_simulada = (circuito, ventana_i)
        grano = f'{len(marcados)} vanos marcados' if marcados else 'todo el circuito'
        cambian = int((resultado['delta_riesgo_ordinal'] != 0).sum())
        avisos = f' | {len(metadata["avisos"])} avisos' if metadata['avisos'] else ''
        STATUS.value = (
            f'{duracion:.2f} s | MIL sobre {metadata["n_vanos"]} bolsas '
            f'({metadata["n_instancias"]:,} eventos) | '
            f'{len(metadata["variables_aplicadas"])} variables aplicadas | '
            f'{cambian} vanos cambian de clase | grafo sobre {grano}'
            f' | importancia ({ranking["n_filas"]:,} muestras){avisos}'
        )
        _redibujar_mapa_predicho()
        _pintar_grafo(grafo)
        _pintar_ranking(ranking)

    aplicar_si_vigente(_escribir, epoca_job=epoca_job, epoca_actual=lambda: _EPOCA)


def _programar_simulacion(*_ignorado):
    global _EPOCA, _tarea_pendiente_simular
    if _tarea_pendiente_simular is not None and not _tarea_pendiente_simular.done():
        _tarea_pendiente_simular.cancel()
    _EPOCA = siguiente_epoca(_EPOCA)
    epoca_job = _EPOCA
    STATUS.value = 'Simulando...'

    async def _tarea():
        try:
            await asyncio.sleep(DEBOUNCE_SEGUNDOS)
        except asyncio.CancelledError:
            return
        _simular(epoca_job)

    _tarea_pendiente_simular = asyncio.ensure_future(_tarea())


boton_simular.on_click(_programar_simulacion)
circuito_widget.observe(_limpiar_resultado_simulacion, names='value')
ventana_widget.observe(_limpiar_resultado_simulacion, names='value')
vano_widget.observe(_redibujar_mapa_predicho, names='value')  # solo redibuja el halo marcado

_redibujar_mapa_predicho()  # primer dibujo: sin simulacion todavia -> "Aun no simulado"
_pintar_grafo(None)


In [ ]:
# --- El panel, ARRIBA y del ancho de la figura (paridad 01.4) -----------------------
# Una sola columna, en el orden en que se usa: circuito -> ventana -> vanos -> variables
# del simulador -> el control de cada variable elegida -> "Simular" -> estado. Cada paso
# depende del anterior, asi que apilarlos evita el zigzag de un flex-wrap donde el boton
# podia quedar antes de los deslizadores que lo alimentan.
#
# El estilo va por CSS y no por `Layout` porque ipywidgets 8 no expone `background`,
# `box-sizing` ni `gap` como traits -- solo `border`, `padding`, `margin` y el flexbox
# basico. `add_class` es la via soportada para lo demas.
ESTILO = widgets.HTML('''
<style>
  .panel-v15 {
    box-sizing: border-box;
    border-radius: 6px; background: #fdf7f6; color: #2b2b2b; font-size: 13px;
  }
  /* Cada grupo, un renglon completo: el panel es una columna, no una grilla. */
  .panel-v15 .grupo-v15 { margin: 0 0 10px 0; width: 100%; }
  .panel-v15 .titulo-v15 { font-weight: 600; margin-bottom: 2px; }
  /* La lista compacta de 01.4: letra 12px, muchas casillas por renglon y scroll propio en
     vez de estirar el panel cuando el circuito tiene cientos de vanos.
     El ancho de cada casilla NO se toca aca: viaja como estilo inline desde su `Layout`
     y le ganaria a esta hoja igual. */
  .lista-vanos, .lista-variables { font-size: 12px; }
  .lista-vanos .widget-checkbox label,
  .lista-variables .widget-checkbox label { white-space: nowrap; font-weight: 400; }
</style>''')


def _grupo(*hijos):
    """Un bloque del panel: su rotulo y sus controles juntos, como los `div` de 01.4."""
    caja = widgets.VBox(list(hijos), layout=widgets.Layout(align_items='flex-start',
                                                           width='100%'))
    caja.add_class('grupo-v15')
    return caja


def _titulo(texto):
    return widgets.HTML(f'<span class="titulo-v15">{texto}</span>')


vano_widget.caja.add_class('lista-vanos')
knob_selector_widget.caja.add_class('lista-variables')

PANEL = widgets.VBox(
    [
        _grupo(_titulo('Circuito'), circuito_widget),
        _grupo(_titulo('Ventana'), ventana_widget),
        _grupo(vano_widget,
               widgets.HBox([boton_marcar_todos, boton_desmarcar]),
               widgets.HTML('<span style="font-size:12px;color:#5b4a48;">Tambien podes '
                            'marcar y desmarcar un vano haciendo clic sobre el en el '
                            'mapa.</span>')),
        _grupo(_titulo('Variables del simulador'), knob_selector_widget),
        _grupo(controles_knob_box),
        _grupo(boton_simular),
        _grupo(STATUS),
    ],
    layout=widgets.Layout(
        width=f'{fig.layout.width}px', align_items='flex-start',
        padding='12px 14px', margin='0 0 6px 0',
        border='1px solid #e4c4c0', border_left='4px solid rgb(203,24,29)',
    ),
)
PANEL.add_class('panel-v15')

APP = widgets.VBox([ESTILO, PANEL, fig],
                   layout=widgets.Layout(width=f'{fig.layout.width}px'))


In [ ]:
display(APP)

## Como leerlo

- **El mapa de ARRIBA (fila 1) es "Criticidad Original"**, el grupo historico calculado
  por 04 sobre eventos observados -- nunca sobre una prediccion. **El de ABAJO (fila 2)
  es "Criticidad Simulada"**, la clase que predice el modelo MGCECDL con las variables
  del simulador aplicadas. No comparten leyenda ni titulo a proposito: mezclarlos
  invitaria a leer una prediccion como si fuera un hecho observado. El tooltip de cada
  tramo lo dice tramo por tramo.
- **Un vano se marca con su casilla o tocandolo en el mapa**, igual que en 04, y las
  dos vias son el mismo estado: el clic alterna la casilla. Los dos mapas aceptan clic
  porque dibujan los mismos vanos. Al elegir circuito el mapa se **encuadra** sobre el
  (centro y zoom salen de su bounding box) y aparecen sus transformadores e
  interruptores; los dos mapas comparten encuadre para que la comparacion arriba/abajo
  mire exactamente la misma geografia.
- **Negro = sin evento, en los DOS mapas.** Un vano sin eventos en la ventana elegida no
  tiene clase, y la ausencia no es el grupo mas bajo. El codigo es el mismo arriba y
  abajo -- negro para la ausencia, los cuatro colores KMeans para las clases -- porque
  tener que recordar dos codigos para la misma cosa es exactamente como se leen mal dos
  mapas apilados. En el mapa simulado el negro cubre ademas lo que quedo fuera de la
  seleccion simulada.
- **"Simular" es el unico disparador** y hace las tres cosas en el mismo job: el mapa
  "Criticidad Simulada" (2 pasadas de bolsas: base + simulado), el grafo reconstruido y el
  panel "Importancia Variables". Aplica solo las variables elegidas en el panel "Variables
  del simulador":
  cada una aparece como un control -- deslizador para numericas, lista desplegable para
  categoricas -- y una familia climatica (precipitacion, temperatura, rafaga y viento)
  se propaga a sus 12 rezagos horarios de una sola vez.
- **"Importancia Variables"** (fila 3, columna 1) NO es SHAP: es un barrido min-max sobre
  el MISMO modelo y la MISMA unidad que el mapa -- las bolsas de los vanos marcados en la
  ventana activa, o del circuito completo si no hay ninguno marcado. Cuesta
  `1 + 2 x controles numericos` pasadas de bolsas (una base compartida y el min/max de
  cada control): medido, 0,09 s para 24 bolsas y 0,34 s para 537, asi que le alcanza con
  el LRU de sesion y no necesita el cache en disco que si pedia la version por filas.
  Se muestra **normalizado con softmax**, asi que las barras suman 100% y se leen como
  participacion relativa; la magnitud cruda de sensibilidad sigue disponible en el hover.
  Ojo con la propiedad del softmax: cuando todas las magnitudes son parecidas las barras
  tienden a repartirse parejo, y eso se lee como "ninguna variable domina", no como un
  error. Con este modelo pasa: sobre las bolsas medidas el barrido mueve el riesgo
  ordinal entre 0,02 y 0,08, asi que las 22 barras quedan cerca del 4,5% que les tocaria
  por reparto uniforme. Es una propiedad del modelo, no del panel.
- **Un vano MARCADO se dibuja con el color de SU clase**, sobre un halo blanco, igual que
  en 04 -- y en **negro** cuando no tiene celda en la ventana (sin eventos, o ausente de
  la tabla). No lleva un color plano de "seleccionado": ese color, encima de la clase,
  congelaba lo que se veia -- la ventana cambiaba la clase por debajo y el vano marcado
  seguia identico en pantalla.
- **El mapa simulado no existe hasta que se presiona "Simular"**: antes de eso la fila 2
  no dibuja nada -- ni tramos, ni equipos, ni leyenda -- y solo muestra el aviso. Un mapa
  completo pintado de "aun no simulado" ocupa el mismo lugar y tiene la misma forma que un
  resultado, que es justo la confusion que esta fila no puede permitirse. Ya dibujado,
  usa el color del grupo KMeans para lo que el simulador predijo -- la misma paleta del
  mapa base -- y **negro** para todo lo demas: vano sin evento en la ventana, o fuera de
  la seleccion simulada.
- **El mapa simulado lo pinta el modelo MIL del cuaderno 05**, que puntua BOLSAS: una
  bolsa es una celda (vano, ventana), sus instancias son los eventos de ese vano en esa
  ventana, y su clase sale de `asignar_clase(n_obs OBSERVADO, u-hat predicho)` sobre la
  geometria KMeans de 04. Por eso los dos mapas comparten paleta **por construccion**: es
  literalmente la misma escala de clases, verificada al cargar el modelo. `n_obs` nunca se
  simula -- es un eje del espacio que define la clase, y moverlo desplazaria el vano por
  una dimension que el modelo no predice.
- **"Grafo reconstruido"** (fila 3, columna 3) es el grafo experto tal como lo usa la
  seleccion activa: `media_vanos(compuerta) x peso_fijo` por arista, en disposicion
  circular, con el peso de cada arista en un marcador sobre su punto medio. Cada nodo
  lleva el **nombre de su variable** y el color de su **modo** -- `climaticos` o
  `estructurales`, la misma particion con la que el modelo fusiona (el clima reescala lo
  estructural). Esa paleta es deliberadamente ajena a la de los grupos KMeans: un rojo en
  el mapa y un rojo en el grafo significarian cosas sin relacion. Se **anula**
  cuando las compuertas no varian entre los vanos de la seleccion (`estadistico_colapso`),
  y eso incluye por construccion cualquier seleccion de menos de 3 vanos: la matriz
  centrada de una o dos filas tiene rango 1. Cuando se anula el panel queda vacio y lo
  dice -- dibujar igual seria presentar el grafo experto FIJO como si lo hubiera estimado
  esta seleccion.
- **"Grupos KMeans de vanos"** (fila 3, columna 2) es la nube de 04 traida aca: cada punto
  es una celda (vano, ventana) en el plano `(eventos, UITI acumulado)`, en el espacio
  canonico `2` (eje Y logaritmico), con los cuatro grupos de fondo, las **fronteras de
  Voronoi** de los centroides pintadas debajo, y las celdas de lo marcado resaltadas
  encima. El fondo es una **muestra fija de 20.000 celdas** de las 111 mil: en un panel de
  ese tamano el resto es sobredibujo, y mandarlas todas por el comm del widget supera el
  limite de datos por segundo del kernel, que descarta el mensaje y deja la figura sin
  dibujar. La muestra es uniforme sobre las filas, asi que las proporciones por grupo se
  mantienen, y la semilla es fija: dos corridas dibujan la misma nube.
  La frontera se calcula con la MISMA funcion que clasifica a los vanos
  (`asignar_clase` sobre una malla), no con una regla propia: un contorno con su propia
  distancia terminaria diciendo una cosa y los puntos de encima otra. El fondo NO depende de la seleccion: 04 ajusta el KMeans una
  sola vez y elegir circuito o vanos solo cambia que se resalta. Es donde se ve directo
  por que mover la ventana cambia la clase de un vano: su punto se desplaza por el plano.
  El panel del grafo reconstruido (decision D4) sigue diferido, con su traza oculta.
- **Cambiar circuito o ventana descarta la ultima simulacion**: la fila 2 vuelve a "Aun no
  simulado" y el panel de importancia se vacia hasta la proxima vez que se presione
  "Simular" -- mostrar la corrida de OTRA seleccion violaria la misma regla
  anti-confusion de arriba.
- **La fila 4 describe a los vanos MARCADOS.** La izquierda es su evolucion temporal:
  una linea de UITI por vano a lo largo de las 11 ventanas, con un corte donde el vano no
  tiene celda -- ese hueco va como vacio y no como cero, porque no es "no hubo UITI" sino
  "no hubo medicion". Se dibujan los primeros 6 vanos marcados, uno por color. Las dos de
  la derecha son el reparto de UITI y de eventos por grupo en la ventana activa. **Sin
  vanos marcados quedan vacias a proposito** (regla de 04): un reparto de miles de vanos y
  otro de tres se dibujan igual y nada en un violin los distingue, asi que caer al circuito
  entero cambiaria el sujeto del panel en silencio.
- **Este cuaderno requiere un kernel Python vivo** -- no es exportable a HTML estatico ni
  publicable como Databricks App.

## La matematica de lo que hace "Simular"

Todo lo de abajo describe UNA pulsacion del boton sobre la seleccion activa
$(c, w, M)$: circuito, ventana y conjunto de vanos marcados.

### 1. Las bolsas de la seleccion

La unidad de prediccion no es el evento: es la **bolsa**, la celda
$(\text{circuito}, \text{vano}, \text{ventana})$. Es la misma unidad en la que 04 define
la criticidad, y por eso el mapa simulado se puede comparar con el historico.

$$\mathcal{B}(c,w,M)=\{\,b=(c,v,w)\;:\;v\in M\,\},\qquad M=\varnothing\;\Rightarrow\;M:=V(c,w)$$

donde $V(c,w)$ son todos los vanos del circuito con al menos un evento en esa ventana:
sin vanos marcados el grano es el circuito completo, no un panel vacio.

Cada bolsa $b$ agrupa sus instancias $I_b$ -- las filas de evento de ese vano dentro de
esa ventana -- y trae dos cosas que **no se predicen nunca**:

$$n_b=|I_b|\quad(\text{eventos OBSERVADOS}),\qquad x_i\in\mathbb{R}^{p},\;p=80$$

Las $p=80$ columnas son 22 estructurales + 48 rezagos de clima + `COD_CAUSA` y sus 9
indicadores. Ojo con no confundir esa cuenta con la **particion por modalidades** que usa
el modelo, que no es la misma: `climaticos` son las 50 columnas de los 48 rezagos **mas
`DDT` y `NR_T`** (descargas y nivel de tormenta son clima, aunque viajen como columnas
estaticas), y `estructurales` son las 30 restantes -- las otras 20 estructurales mas
`COD_CAUSA` y sus 9 indicadores. El almacenamiento es CSR (`offsets`, `counts`), no una matriz rellenada:
el 52,7% de las bolsas son de un solo evento y el maximo es 46, asi que rellenar
desperdiciaria mas de 40x en la mitad de los datos. Al seleccionar, el indice de bolsa se
**renumera** desde 0, porque el modelo toma `n_bags = max(instance_bag)+1` y los ids
originales reservarian una bolsa vacia por cada celda no seleccionada.

**Los controles del simulador** actuan sobre las instancias, no sobre las bolsas. Un
control $\kappa$ gobierna un conjunto de columnas $F(\kappa)$ -- una sola para una
variable estructural, las 12 de una familia climatica -- y aplicarlo es

$$x_{i,j}\;\leftarrow\;\phi_j(\text{valor}),\qquad \forall\, i\in\textstyle\bigcup_b I_b,\;\forall\, j\in F(\kappa)$$

con $\phi_j$ la coercion a espacio de modelo (categoria por su codificador, fecha a
epoch, NaN a su centinela). No hay escalador despues: la matriz de instancias del MIL es
espacio crudo. **$n_b$ jamas se toca**: es un eje del espacio que define la clase, y
moverlo desplazaria al vano por una dimension que el modelo no predice.

### 2. La prediccion de clase de cada vano

El modelo hace **dos pasadas** sobre el mismo codificador. La primera existe solo para
producir las compuertas del grafo.

**(a) Codificacion y atencion.** Cada instancia se codifica por modalidad
(30 estructurales, 50 climaticas, segun la particion de arriba) y se concatena en
$z_i^{(1)}$. La bolsa se resume con
atencion tipo Ilse, normalizada dentro de la bolsa:

$$e_i=\mathbf{w}^{\top}\tanh(V z_i^{(1)}),\qquad
a_i=\frac{\exp(e_i)}{\sum_{i'\in I_b}\exp(e_{i'})},\qquad
z_b^{(1)}=\sum_{i\in I_b}a_i\,z_i^{(1)}$$

Esto es **invariante a la cardinalidad por construccion**: duplicar cada instancia de una
bolsa no cambia ningun $e_i$, el denominador se duplica, cada copia recibe $a_i/2$ y la
suma queda igual.

**(b) Compuertas del grafo experto.** Un decodificador lee el resumen de la bolsa y
produce una compuerta por arista:

$$g_b=2\,\sigma(W_g\,z_b^{(1)})\;\in\;(0,2)^{E},\qquad E=64$$

Se inicializa en cero, de modo que $g_b=\mathbf{1}$ al arrancar y el grafo aprendido
**parte exactamente del grafo experto fijo**.

**(c) Propagacion.** El grafo experto es una adyacencia fija $W\in\mathbb{R}^{80\times 80}$
con soporte en $E$ aristas. Cada instancia recibe, en la columna destino de cada arista:

$$x'_{i,j}\;=\;x_{i,j}\;+\;\alpha\!\!\sum_{e\,:\,\mathrm{dst}(e)=j}\!\! g_{b(i),e}\;w_e\;x_{i,\mathrm{src}(e)},
\qquad \alpha=0{,}2$$

Una columna que no es destino de ninguna arista queda intacta, exactamente.

**(d) Segunda pasada y fusion FiLM.** El MISMO codificador procesa $x'$, se vuelve a
agrupar con la MISMA atencion y las dos modalidades se fusionan modulando la
estructural con la climatica (`film_modulated_modality = estructurales`, del artefacto):

$$\hat z_b=z_b^{\text{est}}\odot\bigl(1+\gamma(z_b^{\text{clim}})\bigr)+\beta(z_b^{\text{clim}}),
\qquad p_b=h(\hat z_b),\qquad \hat u_b=\mathrm{expm1}(p_b)$$

La concatenacion es aditiva entre modalidades y no puede representar un producto entre una
variable estructural y una climatica; FiLM hace que el clima **reescale** lo estructural,
que es tambien la afirmacion de dominio: una rafaga pesa mas sobre un apoyo alto, viejo y
degradado. El modelo aprende en $\log(1+u)$ y se devuelve a UITI con `expm1`.

**(e) Clase.** Con el UITI predicho y los eventos observados, la clase sale de la
geometria KMeans de 04 -- **la misma que pinta el mapa base**, verificada al cargar el
modelo. En el espacio canonico `2` el eje de eventos es lineal y el de UITI logaritmico:

$$\zeta_b=\left(\frac{n_b-\mu_0}{s_0},\;\frac{\log_{10}\max(\hat u_b,\varepsilon)-\mu_1}{s_1}\right),
\qquad \hat k_b=\arg\min_{k\in\{0,1,2,3\}}\lVert \zeta_b-c_k\rVert^2$$

El mapa pinta $\hat k_b$ con la paleta de los cuatro grupos; lo que no tiene bolsa en la
ventana, o quedo fuera de la seleccion, va en negro. El simulador corre esto **dos veces**
-- sin y con los controles aplicados -- y $\Delta_b=\hat k_b^{\text{sim}}-\hat k_b^{\text{base}}$
es cuantos vanos cambian de clase.

### 3. El ranking de importancia de variables

Es un barrido min-max, **no SHAP**. Se mide sobre las mismas bolsas del mapa, con el
riesgo ordinal esperado: la distribucion suave sobre las cuatro clases, promediada.

$$P_{b,k}=\frac{\exp(-\lVert\zeta_b-c_k\rVert^2/\tau)}{\sum_{k'}\exp(-\lVert\zeta_b-c_{k'}\rVert^2/\tau)},
\qquad
R(X)=\frac{1}{|\mathcal{B}|}\sum_{b}\sum_{k}k\,P_{b,k}\;\in[0,3]$$

Para cada control **numerico** $\kappa$, con su rango observado $[m_\kappa,M_\kappa]$ en
todo el dataset, se lleva a sus dos extremos todas las columnas $F(\kappa)$ a la vez:

$$\Delta^{-}_{\kappa}=R\!\left(X^{\kappa\to m_\kappa}\right)-R(X),\qquad
\Delta^{+}_{\kappa}=R\!\left(X^{\kappa\to M_\kappa}\right)-R(X),\qquad
s_\kappa=\max\!\left(|\Delta^{-}_{\kappa}|,\,|\Delta^{+}_{\kappa}|\right)$$

Los controles categoricos y constantes **se omiten**: no tienen minimo/maximo numerico, e
inventarles un rango seria puntuar un escenario que nadie pidio (por eso el panel muestra
22 barras de 26 controles). Las barras son la normalizacion softmax, estable por resta del
maximo:

$$\rho_\kappa=\frac{\exp\bigl((s_\kappa-\max_{\kappa'}s_{\kappa'})/T\bigr)}{\sum_{\kappa'}\exp\bigl((s_{\kappa'}-\max s)/T\bigr)},\qquad T=1,\qquad \sum_\kappa\rho_\kappa=1$$

**Cuidado al leerlas**: el softmax aplana. Con este modelo el barrido mueve el riesgo
ordinal entre 0,02 y 0,08, asi que las 22 barras quedan cerca del 4,5% del reparto
uniforme. Eso dice "ninguna variable domina", y es una propiedad del modelo, no del panel.
La magnitud cruda $s_\kappa$ sigue en el hover.

### 4. El grafo inferido

Las compuertas de la parte (b) son lo unico del grafo que depende de la seleccion. Se
juntan en una matriz $G\in\mathbb{R}^{|\mathcal{B}|\times E}$, con $G_{b,e}=g_{b,e}$.

Antes de reconstruir nada se mide si esas compuertas **varian** entre vanos. Con
$\tilde G$ la matriz centrada por columnas y $\sigma_1,\dots$ sus valores singulares:

$$\mathrm{var}=\frac{1}{E}\sum_e \mathrm{Var}_b(G_{b,e}),\qquad
\mathrm{rank}_{\text{ef}}=\frac{\bigl(\sum_r\sigma_r^2\bigr)^2}{\sum_r\sigma_r^4},\qquad
\text{colapso}\iff \max_e \mathrm{std}_b(G_{b,e})<10^{-6}\;\lor\;\mathrm{rank}_{\text{ef}}\le 1$$

El rango efectivo es el cociente de participacion: vale $\approx 1$ cuando toda la
variacion vive en una sola direccion, es decir, cuando todos los vanos estan compuertados
igual. Un colapso **anula** el grafo y el panel lo dice, en vez de dibujar el grafo
experto fijo como si lo hubiera estimado esta seleccion. De ahi sale un limite duro:
con $|\mathcal{B}|<3$ la matriz centrada tiene rango 1 por construccion, asi que **menos
de 3 vanos nunca producen grafo**.

Si no hay colapso, el peso reconstruido de cada arista es el peso experto fijo tal como lo
usa esta familia de vanos:

$$\bar g_e=\frac{1}{|\mathcal{B}|}\sum_{b}G_{b,e},\qquad
A_{\mathrm{src}(e),\,\mathrm{dst}(e)}=\bar g_e\cdot w_e,\qquad A_{ij}=0 \text{ fuera del soporte}$$

El panel lo dibuja en disposicion circular sobre las variables que participan de al menos
una arista, con $A_{ij}$ en un marcador sobre el punto medio de cada arista.

### Presupuesto de una pulsacion

| Paso | Pasadas de bolsas |
|---|---|
| Mapa simulado (base + simulado) | 2 |
| Compuertas para el grafo | 1 |
| Importancia (base compartida + min/max por control) | $1+2K$, $K=22$ |
| **Total** | **48** |

Medido sobre el modelo real: 0,10 s para 24 bolsas, y 0,34 s solo de importancia para 537.
Por eso alcanza con un LRU de sesion y no hace falta cache en disco.